# Pre-SFT Baseline: Context-Parametric Inversion

Establishes the true step-0 anchor for the CPI trajectory: evaluates the raw
pretrained base model (no LoRA adapter -- the exact pre-SFT state both the
`alpaca` and `tulu` runs started from) on the conflict-eval set.

Uses `eval.py --logprob-only`: Layer 0 (filter) + Layer 1 (`method_logprob`)
are both teacher-forced log-prob comparisons over the candidate answers, so
neither needs the model to generate free-form text. A raw base model can't
reliably follow the "answer in a few words" instruction -- `--logprob-only`
skips generation entirely rather than trying to parse noisy generated text
that was never load-bearing for `R_ctx`/`R_par` to begin with.

## 0. Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Get the codebase (`dev` branch)

Clones fresh if not already present; otherwise fetches and fast-forwards to
the latest `dev`. `dev` is where active work lands (the judge-fix-only
correction + this baseline's `--logprob-only` flag are both already there).

In [13]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull --ff-only origin {BRANCH}

%cd {REPO_DIR}
!git log --oneline -3
print("\nRepo dir:", REPO_DIR)

From https://github.com/GIRIAYUSH/context-parametric-inversion-research
 * branch            dev        -> FETCH_HEAD
Already on 'dev'
Your branch is up to date with 'origin/dev'.
From https://github.com/GIRIAYUSH/context-parametric-inversion-research
 * branch            dev        -> FETCH_HEAD
Already up to date.
/content/context-parametric-inversion-research
9123e48 (HEAD -> dev, origin/dev) notebook: add judge-escalation pass (section 4b) for the base-model AMBIG items
b41b90b Merge master (--judge-prompt-version fix) into dev
4c98bf6 eval.py: default judge to v2 prompt (--judge-prompt-version, default v2)

Repo dir: /content/context-parametric-inversion-research


## 2. Install dependencies

In [11]:
!pip install -q -U transformers accelerate huggingface_hub openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 79.9 MB/s eta 0:00:00


## 3. Configuration -- checkpoint locations on Drive + secrets

`CPI_CKPT_DIR_ALPACA` / `CPI_CKPT_DIR_TULU` point at the same Drive folders
used elsewhere (each directly contains `checkpoint-<step>/` and `final/`) --
not needed for the baseline eval itself (no adapter is loaded), but recorded
here so this notebook is self-contained for whatever comes after the
baseline run.

Secrets are pulled from Colab's secrets manager (the key icon in the left
sidebar) via `userdata.get(...)`, never pasted inline -- this notebook lives
under `experiment-notebooks/` and IS tracked by git, unlike `src/runs.ipynb`.

In [14]:
import os
from google.colab import userdata
from huggingface_hub import login

# <-- EDIT if your Drive paths differ
os.environ["CPI_CKPT_DIR_ALPACA"] = "/content/drive/MyDrive/Checkpoints - Alpaca - CPI- Analysis/Alpaca_Checkpoints/checkpoints"
os.environ["CPI_CKPT_DIR_TULU"]   = "/content/drive/MyDrive/Checkpoints - TULU - CPI- Analysis/checkpoints"

# Base model both runs actually trained on -- congif.yaml's run.model / models.<name>.hf_id
BASE_MODEL = "meta-llama/Llama-2-7b-hf"  # <-- EDIT if different

# Add HF_TOKEN and OPENAI_API_KEY as Colab secrets first (key icon, left
# sidebar) -- HF_TOKEN for the gated Llama-2 weights, OPENAI_API_KEY for the
# Layer-2 judge (only needed for section 4b below).
login(token=userdata.get("HF_TOKEN"))
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("CPI_CKPT_DIR_ALPACA:", os.environ["CPI_CKPT_DIR_ALPACA"])
print("CPI_CKPT_DIR_TULU  :", os.environ["CPI_CKPT_DIR_TULU"])
print("BASE_MODEL         :", BASE_MODEL)


## 4. Run the pre-SFT baseline eval (Layers 0+1 only, no generation)

Loads the raw pretrained base model -- no adapter applied, i.e. the true
step-0 state -- with `use_chat_template=True` (default), matching every
existing checkpoint's recorded `config`. Writes
`results/cpi-results/presft-baseline/Llama-2-7b-hf_eval.json` in the same
schema as `results/phase0-results/model_*/results.json`.

In [15]:
!python src/evaluation/eval.py \
    --model-id {BASE_MODEL} \
    --dataset dataset/conflict_eval_unified.json \
    --output-dir results/cpi-results/presft-baseline-judged \
    --use-judge \
    --judge-prompt-version v2

Loaded 417 total | dropped 5 flagged item(s) | 412 usable
Loading meta-llama/Llama-2-7b-hf  precision=bf16
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [00:04<00:00, 69.84it/s]
  Memory footprint: 13.48 GB (precision=bf16)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=24) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hug

## 5. Copy the result to Drive (optional, keeps it if the Colab runtime resets)

In [ ]:
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/cpi-presft-baseline"  # <-- EDIT if you want a different Drive spot
!mkdir -p "{DRIVE_RESULTS_DIR}"
!cp -v results/cpi-results/presft-baseline/*.json "{DRIVE_RESULTS_DIR}/"